<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1 = q('''
SELECT a.name, a.country
FROM tracks t
JOIN artists a ON a.artist_id = t.artist_id
''')

print(q1)

             name country
0      Nova Waves      US
1      Nova Waves      US
2  The Blue Ridge      US
3         Kestrel      UK
4         Kestrel      UK
5         Marisol      ES
6  The Blue Ridge      US
7  The Blue Ridge      US
8         Kestrel      UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT t.genre, AVG(t.seconds)
FROM tracks t
WHERE t.genre IS NOT NULL
GROUP BY t.genre
ORDER BY AVG(t.seconds) DESC
''')
# genre with longest track is Electronic

,genre,AVG(t.seconds)
0,Electronic,287.5
1,Pop,220.5
2,Latin,210.0
3,Folk,203.0


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT p.user, COUNT(p.play_id) as plays, COUNT(DISTINCT p.track_id)
FROM plays p
GROUP BY p.user
''')
print(q3)

   user  plays  COUNT(DISTINCT p.track_id)
0   ava      4                           4
1   ben      3                           3
2  cara      2                           2
3   dan      2                           2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4 = q('''
SELECT t.title
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.play_id IS NULL
''')
print(q4)

           title
0      Ridgeline
1  Untitled Demo


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT a.name, ROUND((SUM(t.seconds)/ 60), 1) AS minutes
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON  t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY minutes DESC
''')

,name,minutes
0,Kestrel,19.0
1,Nova Waves,14.0
2,The Blue Ridge,6.0
3,Marisol,3.0


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT t.track_id, t.title
FROM tracks t
WHERE t.genre IS NULL
''')

# If we would've done WHERE genre != 'Pop', the untagged row wouldn't have
# appeared because NULL isn't treated as a value type so you can't compare it
# using != in SQL

,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT p.played_on, COUNT(DISTINCT p.user), COUNT(p.played_on)
FROM plays p
GROUP BY p.played_on
ORDER BY p.played_on
''')

,played_on,COUNT(DISTINCT p.user),COUNT(p.played_on)
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

My biggest issue came down to the validation check, I wasn't understanding why I wasn't passing q3. Even after reading the stack trace I was still a bit confused until I realized that it was looking for a column specifically named plays, and in my original code I didn't create a column named plays. When I was counting the rows of the plays table I never created an alias for it, and realized that in the validation check it was looking for an alias. So I had to go back and add an alias to COUNT(p.play_id), so that it would now be COUNT(p.play_id) AS plays.